# 消融：去掉 `is_amount_75_110` 是否更好？

对比定稿组合（`confirm_winner.json`）：

- **A（基准）**：`IF+Ed[log1p+hours+micro+bands]+A_top2(V4+V14)+FE8[abs_v14_minus_v12+v14_x_log1p_amount+v4_minus_v14]`
- **B（去掉 75–110 带）**：同上，但 EDA 中移除 `is_amount_75_110`，仅保留 `is_amount_1_30`

**评估协议**
- purged walk-forward 5 折 CV（与主 notebook 一致）
- 超参：§12 `12_tune/tune_results.json`（Stage2 定稿超参）
- 种子：`CONFIRM_SEEDS` = [7, 13, 123, 256, 3141]
- 模型：LightGBM + XGBoost

> 首次运行会计算/缓存 `if_oof_score`（约数分钟），之后从 `output/final_report/cache/if_oof_score.npy` 读取。

In [3]:
%matplotlib inline
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
SRC_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'experiments' else NOTEBOOK_DIR
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from experiments.winner_cv_utils import (
    EDA_NO_75,
    EDA_WINNER,
    build_winner_dataframe,
    find_project_root,
    load_confirm_seeds,
    load_tune_params,
    run_ablation_matrix,
    summarize_ablation,
    winner_feature_cols,
)

PROJECT_ROOT = find_project_root(SRC_DIR)
OUT_DIR = PROJECT_ROOT / 'src' / 'output' / 'experiments' / 'drop_amount_75_110'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('项目根目录:', PROJECT_ROOT)
print('输出目录:', OUT_DIR)

项目根目录: /Users/jingyuhe/Desktop/credit-fraud-dealing-with-imbalanced-datasets
输出目录: /Users/jingyuhe/Desktop/credit-fraud-dealing-with-imbalanced-datasets/src/output/experiments/drop_amount_75_110


In [4]:
VARIANTS = {
    'A_bands_full': EDA_WINNER,
    'B_drop_is_amount_75_110': EDA_NO_75,
}

LABEL_A = 'IF+Ed[log1p+hours+micro+bands]+A_top2(V4+V14)+FE8[abs_v14_minus_v12+v14_x_log1p_amount+v4_minus_v14]'
LABEL_B = 'IF+Ed[log1p+hours+micro+is_amount_1_30]+A_top2(V4+V14)+FE8[abs_v14_minus_v12+v14_x_log1p_amount+v4_minus_v14]'
DISPLAY = {'A_bands_full': LABEL_A, 'B_drop_is_amount_75_110': LABEL_B}

tune = load_tune_params(PROJECT_ROOT)
seeds = load_confirm_seeds(PROJECT_ROOT)
print('§12 Stage2 超参:')
for m in ['LightGBM', 'XGBoost']:
    print(f'  {m}:', {k: v for k, v in tune[m].items() if k != 'feature_group'})
print('CONFIRM_SEEDS:', seeds)

§12 Stage2 超参:
  LightGBM: {'learning_rate': 0.01, 'weight_scheme': 'no_weight', 'max_depth': 6, 'num_leaves': 15, 'min_child_samples': 10, 'subsample': 0.6, 'colsample_bytree': 0.9, 'reg_alpha': 0.1, 'reg_lambda': 0.1}
  XGBoost: {'learning_rate': 0.07, 'weight_scheme': 'spw_0.5x', 'max_depth': 3, 'min_child_weight': 1, 'subsample': 0.9, 'colsample_bytree': 1.0, 'reg_alpha': 0.1, 'reg_lambda': 1.0}
CONFIRM_SEEDS: [7, 13, 123, 256, 3141]


In [5]:
df, base_features = build_winner_dataframe(PROJECT_ROOT, cache_if=True)
print(f'样本: {len(df):,} | 欺诈率: {df["Class"].mean():.4f}')
for key, eda in VARIANTS.items():
    cols = winner_feature_cols(base_features, eda)
    print(f'{key}: {len(cols)} 列 | EDA={eda}')

compute if_oof_score (purged OOF IF)...
  IF fold 1/5 done, normal_train=47,326
  IF fold 2/5 done, normal_train=50,000
  IF fold 3/5 done, normal_train=50,000
  IF fold 4/5 done, normal_train=50,000
  IF fold 5/5 done, normal_train=50,000
cached -> /Users/jingyuhe/Desktop/credit-fraud-dealing-with-imbalanced-datasets/src/output/final_report/cache/if_oof_score.npy
样本: 284,807 | 欺诈率: 0.0017
A_bands_full: 41 列 | EDA=['log1p_amount', 'hours_since_start', 'is_micro_testing', 'is_amount_1_30', 'is_amount_75_110']
B_drop_is_amount_75_110: 40 列 | EDA=['log1p_amount', 'hours_since_start', 'is_micro_testing', 'is_amount_1_30']


In [6]:
raw = run_ablation_matrix(df, base_features, VARIANTS, seeds, tune)
raw['combo_label'] = raw['variant'].map(DISPLAY)
raw.to_csv(OUT_DIR / 'ablation_raw.csv', index=False, encoding='utf-8-sig')
summary = summarize_ablation(raw)
summary['combo_label'] = summary['variant'].map(DISPLAY)
summary.to_csv(OUT_DIR / 'ablation_summary.csv', index=False, encoding='utf-8-sig')
display(summary)

[1/20] A_bands_full | LightGBM | seed=7
[2/20] A_bands_full | LightGBM | seed=13
[3/20] A_bands_full | LightGBM | seed=123
[4/20] A_bands_full | LightGBM | seed=256
[5/20] A_bands_full | LightGBM | seed=3141
[6/20] A_bands_full | XGBoost | seed=7
[7/20] A_bands_full | XGBoost | seed=13
[8/20] A_bands_full | XGBoost | seed=123
[9/20] A_bands_full | XGBoost | seed=256
[10/20] A_bands_full | XGBoost | seed=3141
[11/20] B_drop_is_amount_75_110 | LightGBM | seed=7
[12/20] B_drop_is_amount_75_110 | LightGBM | seed=13
[13/20] B_drop_is_amount_75_110 | LightGBM | seed=123
[14/20] B_drop_is_amount_75_110 | LightGBM | seed=256
[15/20] B_drop_is_amount_75_110 | LightGBM | seed=3141
[16/20] B_drop_is_amount_75_110 | XGBoost | seed=7
[17/20] B_drop_is_amount_75_110 | XGBoost | seed=13
[18/20] B_drop_is_amount_75_110 | XGBoost | seed=123
[19/20] B_drop_is_amount_75_110 | XGBoost | seed=256
[20/20] B_drop_is_amount_75_110 | XGBoost | seed=3141


,variant,model,AUC_mean,AUC_std,F1_mean,F1_std,FP_mean,FN_mean,n_runs,combo_label
0,A_bands_full,LightGBM,0.784469,0.005058,0.823582,0.008189,33.6,80.2,5,IF+Ed[log1p+hours+micro+bands]+A_top2(V4+V14)+...
1,A_bands_full,XGBoost,0.791726,0.004074,0.822016,0.003889,35.2,80.0,5,IF+Ed[log1p+hours+micro+bands]+A_top2(V4+V14)+...
2,B_drop_is_amount_75_110,LightGBM,0.783425,0.005679,0.817603,0.006892,36.4,81.6,5,IF+Ed[log1p+hours+micro+is_amount_1_30]+A_top2...
3,B_drop_is_amount_75_110,XGBoost,0.790809,0.005504,0.821424,0.005659,32.8,82.0,5,IF+Ed[log1p+hours+micro+is_amount_1_30]+A_top2...


In [7]:
# Δ(B - A)：正数表示去掉 is_amount_75_110 后指标提升
pivot_auc = summary.pivot(index='variant', columns='model', values='AUC_mean')
pivot_f1 = summary.pivot(index='variant', columns='model', values='F1_mean')
delta_rows = []
for model in pivot_auc.columns:
    delta_rows.append({
        'model': model,
        'Δ_AUC-PR (B-A)': pivot_auc.loc['B_drop_is_amount_75_110', model] - pivot_auc.loc['A_bands_full', model],
        'Δ_F1@best (B-A)': pivot_f1.loc['B_drop_is_amount_75_110', model] - pivot_f1.loc['A_bands_full', model],
        'A_含75_110_AUC': pivot_auc.loc['A_bands_full', model],
        'B_无75_110_AUC': pivot_auc.loc['B_drop_is_amount_75_110', model],
        'A_含75_110_F1': pivot_f1.loc['A_bands_full', model],
        'B_无75_110_F1': pivot_f1.loc['B_drop_is_amount_75_110', model],
    })
delta = pd.DataFrame(delta_rows)
delta.to_csv(OUT_DIR / 'ablation_delta_B_minus_A.csv', index=False, encoding='utf-8-sig')
display(delta.round(4))

better_auc = (delta['Δ_AUC-PR (B-A)'] > 0).all()
better_f1 = (delta['Δ_F1@best (B-A)'] > 0).all()
if better_auc and better_f1:
    print('结论：两模型 AUC-PR 与 F1 均提升 → 可尝试去掉 is_amount_75_110。')
elif (delta['Δ_AUC-PR (B-A)'] < 0).all() and (delta['Δ_F1@best (B-A)'] < 0).all():
    print('结论：去掉 is_amount_75_110 后指标下降 → 保留原 bands。')
else:
    print('结论：两模型或两指标不一致，请结合 FP/FN 与 Δ 表综合判断。')

,model,Δ_AUC-PR (B-A),Δ_F1@best (B-A),A_含75_110_AUC,B_无75_110_AUC,A_含75_110_F1,B_无75_110_F1
0,LightGBM,-0.0010,-0.0060,0.7845,0.7834,0.8236,0.8176
1,XGBoost,-0.0009,-0.0006,0.7917,0.7908,0.8220,0.8214


结论：去掉 is_amount_75_110 后指标下降 → 保留原 bands。
